# 10 — Démonstration Gradio

**Objectif :** valider l'interface locale sans lancer un serveur bloquant pendant l'exécution.

**Entrée :** bundle du notebook 09.  
**Sortie :** rapport de smoke test Gradio.  
**Dépendance :** notebook 09.  
**Temps estimé :** 1 à 2 minutes.  
**Ressources :** CUDA pour l'inférence.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.app import build_demo
from src.recommender import HybridRecommender

recommender = HybridRecommender(ROOT / "artifacts")
rows = recommender.recommend("open world car racing", k=5)
display(pd.DataFrame(rows))

demo = build_demo(recommender)
config = demo.get_config_file()
assert config["mode"] == "blocks"

report = {
    "status": "ok",
    "device": recommender.device,
    "result_count": len(rows),
    "unique_skus": len({row["sku"] for row in rows}),
    "launch_command": ".\\.venv\\Scripts\\python.exe -m src.app",
    "url": "http://127.0.0.1:7860",
}
(ROOT / "reports" / "gradio_smoke.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(report, indent=2, ensure_ascii=False))

## Lancement interactif

Dans PowerShell :

```powershell
.\.venv\Scripts\python.exe -m src.app
```

Puis ouvrir <http://127.0.0.1:7860>. `share=False` empêche la création d'un tunnel public.